## Installation Check
You should be able to run all cells to advance with the course.

Make sure you selected the EXTERNAL jupyter server in PyCharm (top right corner).
Every cell has some troubleshooting guidelines included.

### Step 1 : Importing needed modules
Possible error solutions:
1. Check if you are using the EXTERNAL jupyter server (top right corner in PyCharm). If not, you are trying to run locally and your local system is not setup for Spark.
2. If you applied some changes to the environment during this session, restart the PyCharme IDE.

In [11]:
from pyspark.sql import SparkSession

### Step 2 : Setting up ConnectionConfig
ConnectionConfig helps you with connection stuff. It is important that this script is avalaible in jupyter.
cc.listEnvironment() will show you the environment variables that are set on the jupyter server.

Possible error solutions:
1. If you get an error importing ConnectionConfig, go through the data architecture setup instructions again. Make sure you set the path to your project correct in deployDataArchitecture. The projectdirectory is mounted to /home/jovyan/work in the jupyter server. If this is not correct, jupyter will not find ConnectionConfig.py


In [21]:
import ConnectionConfig as cc
#cc.setupEnvironment()
cc.listEnvironment()

SHELL: /bin/bash
CONDA_EXE: /opt/conda/bin/conda
_CE_M: 
HOSTNAME: sparkjupyter
LANGUAGE: C.UTF-8
_START_SH_EXECUTED: 1
SPARK_OPTS: --driver-java-options=-Xms1024M --driver-java-options=-Xmx4096M --driver-java-options=-Dlog4j.logLevel=info
NB_UID: 1000
XML_CATALOG_FILES: file:///opt/conda/etc/xml/catalog file:///etc/xml/catalog
PWD: /home/jovyan
CONDA_PREFIX: /opt/conda
PYSPARK_SUBMIT_ARGS: --packages io.delta:delta-spark_2.13:4.0.0 pyspark-shell
HOME: /home/jovyan
LANG: C.UTF-8
NB_GID: 100
CONDA_PROMPT_MODIFIER: (base) 
_CONDA_EXE: /opt/conda/bin/conda
_CONDA_ROOT: /opt/conda
_CE_CONDA: 
DELTA_VERSION: 4.0.0
CONDA_SHLVL: 1
SHLVL: 0
CONDA_DIR: /opt/conda
SPARK_HOME: /usr/local/spark
CONDA_PYTHON_EXE: /opt/conda/bin/python
JUPYTER_PORT: 8888
SPARK_CONF_DIR: /usr/local/spark/conf
CONDA_DEFAULT_ENV: base
NB_USER: jovyan
LC_ALL: C.UTF-8
PATH: /opt/conda/bin:/opt/conda/condabin:/usr/local/spark/bin:/opt/conda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/local/spark/

### Step 3 : Configuring the sparkSession
Possible error solutions:
1. Make sure the imports in step 1 succeeded.

In [22]:
builder = SparkSession.builder \
    .appName("InstallCheck") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "4") \
    .master("local[*]")

### Step 4  : Creating a local spark cluster
This step starts the sparkSession. Because you are running a local cluster.
If you already started a sparkSession with getOrCreate(), running this cell does not change the session. Restart the Jupyter server, and rerun all above cells again
After running this step you will get the url (click on Spark UI) to the Spark server. Check if you can visit the URL

Possible error solutions:

1. Make sure the previous step was executed correctly
2. Read the error message. If you don't get a clear error message look at sparkjupyter container log. The console will give information about the startup proces of the Spark-server.

In [23]:
spark = builder.getOrCreate()

In [24]:

spark.getActiveSession()

### Step 5  : Reading source into Spark DataFrame

Possible error solutions:
1. Make sure the file is present in the project at [file_location]

In [25]:
# File location and type
file_location = "./FileStore/tables/shakespeare.txt"
file_type = "text"

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type)  \
  .load(file_location)
df.show()
df.describe()

+--------------------+
|               value|
+--------------------+
|This is the 100th...|
|is presented in c...|
|Library of the Fu...|
|often releases Et...|
|                    |
|         Shakespeare|
|                    |
|*This Etext has c...|
|                    |
|<<THIS ELECTRONIC...|
|SHAKESPEARE IS CO...|
|PROVIDED BY PROJE...|
|WITH PERMISSION. ...|
|DISTRIBUTED SO LO...|
|PERSONAL USE ONLY...|
|COMMERCIALLY.  PR...|
|SERVICE THAT CHAR...|
|                    |
|*Project Gutenber...|
|in the presentati...|
+--------------------+
only showing top 20 rows


DataFrame[summary: string, value: string]

### Step 7  : Creating a view on the source and performing SQL on View
This step should not pose any problem if the previous steps where successful.


In [26]:
df.createOrReplaceTempView('lines')
words = spark.sql('select explode(split(value, " ")) from lines')
words.createOrReplaceTempView('words')
lowerwords = spark.sql('select lower(trim(col)) as word, count(*) as amount from words where lower(trim(col)) <> "" group by lower(trim(col)) order by amount desc limit 20')
lowerwords.show()


+----+------+
|word|amount|
+----+------+
| the| 27549|
| and| 26037|
|   i| 19540|
|  to| 18700|
|  of| 18010|
|   a| 14383|
|  my| 12455|
|  in| 10671|
| you| 10630|
|that| 10487|
|  is|  9145|
| for|  7982|
|with|  7931|
| not|  7643|
|your|  6871|
| his|  6749|
|  be|  6700|
| but|  5886|
|  he|  5884|
|  as|  5882|
+----+------+



### Step 8  : Saving the result as a Delta table
After running this step you should have a directory spark-warehouse/shakespeareWords in your project directory. This directory contains the Delta table. Right click the root directory and click "Reload all from disk" to see the directory.

Possible error solutions:
1. Make sure the previous step was executed correctly
2. Make sure you have the correct permissions to write to the project directory.

In [27]:
# With this registered as a temp view, it will only be available to this particular notebook. If you'd like other users to be able to query this table, you can also create a table from the DataFrame.
# Once saved, this table will persist across cluster restarts as well as allow various users across different notebooks to query this data.
# To do so, choose your table name and uncomment the bottom line.
lowerwords.describe()

permanent_table_name = "shakespeareWords"

lowerwords.write.format("delta").mode("overwrite").saveAsTable(permanent_table_name)

### Step 9  : Stopping the Spark session
After running a notebook always stop the spark session. This will free resources on the jupyter server.
!!! Also shut down the external jupyter kernel in PyCharm (top right corner). This frees up memory the docker container uses on your local system. !!!

In [28]:
spark.stop()